In [2]:
import pandas as pd

fraud = pd.read_csv(r'C:\Users\Admin\Downloads\Datasets_For_Fraud_Detection.csv')
fraud.sample(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
445977,19,CASH_OUT,169534.76,C519305523,50197.00,0.00,C84868863,0.00,169534.76,0,0
278156,164,TRANSFER,135907.36,C1645500313,0.00,0.00,C1233092832,2298692.23,2434599.59,0,0
23841,212,PAYMENT,2239.01,C586247272,53004.00,50764.99,M1441400069,0.00,0.00,0,0
74045,277,CASH_OUT,363673.00,C810507376,90248.00,0.00,C903490106,0.00,363673.00,0,0
561910,251,CASH_IN,20500.67,C929919765,2164182.32,2184682.99,C1288331313,170993.63,150492.96,0,0
464651,331,CASH_OUT,371887.05,C873449210,0.00,0.00,C1030928319,2897432.60,3269319.65,0,0
449898,13,PAYMENT,9429.91,C840111600,0.00,0.00,M1599202954,0.00,0.00,0,0
121249,297,TRANSFER,1006340.20,C1307619532,0.00,0.00,C1129835410,1272152.25,2490152.19,0,0
48643,308,CASH_OUT,304550.61,C626891144,0.00,0.00,C864145030,4686313.22,4990863.84,0,0
342486,306,CASH_IN,1825.29,C1386048016,32773.00,34598.29,C1660525945,807620.00,805794.71,0,0


## Handling Data Inconsistencies in the Dataset

During the exploratory phase, it was observed that a significant portion of the dataset contains transactions that do not follow logical financial rules. For example, some transactions show zero account balances while still allowing withdrawals, or inconsistencies between expected and actual balances after a transaction.

Despite these inconsistencies, the decision was made **not to remove these records** before training the model. This decision was based on the following considerations:

### 1. Preserving Data Volume

Approximately 63% of the dataset was identified as containing such inconsistencies. Removing these rows would result in a substantial loss of data, which could negatively impact the model’s ability to learn meaningful patterns.

### 2. Reflecting Real-World Scenarios

In real-world financial systems, data is often imperfect and may contain anomalies due to system errors, delays, or incomplete information. Retaining these records allows the model to be exposed to such irregularities, making it more robust and realistic.

### 3. Leveraging Feature Engineering

Instead of removing inconsistent data, a feature engineering approach was adopted. New variables such as `balance_error` and `is_logical` were created to explicitly capture these inconsistencies. This allows the model to learn from them rather than ignore them.

### 4. Avoiding Bias

Dropping a large portion of the dataset could introduce bias, especially if fraudulent transactions are disproportionately represented within the inconsistent records.

### Conclusion

Rather than discarding a large portion of the dataset, the approach taken was to retain all transactions and incorporate data quality indicators as features. This ensures that the model benefits from the full dataset while still accounting for inconsistencies in a structured and interpretable manner.


## Feature Engineering

Feature engineering was performed to create additional variables that better represent transaction behaviour.

The original dataset contains basic transaction details, but some important fraud patterns are not directly represented. Therefore, new features were created to capture balance inconsistencies and suspicious transaction behaviours.

The following features were added:

- `balance_error`: Measures the difference between the expected and actual new balance after a transaction.
- `is_full_drain`: Identifies transactions where the sender's account balance is completely emptied.
- `is_logical`: Indicates whether a transaction follows expected financial balance behaviour (1 = logical, 0 = not logical).

These engineered features provide additional information that can help machine learning models identify fraudulent transactions.

In [3]:
fraud['balance_error'] = (
    fraud['oldbalanceOrg'] 
    - fraud['amount'] 
    - fraud['newbalanceOrig']
)

fraud['is_full_drain'] = (
    fraud['newbalanceOrig'] == 0
).astype(int)

def check_logical(row):

    # Check sender balance consistency
    if abs((row['oldbalanceOrg'] - row['amount']) - row['newbalanceOrig']) > 1e-2:
        return 0

    # CASH_OUT without available balance
    if row['type'] == 'CASH_OUT' and row['oldbalanceOrg'] == 0:
        return 0

    # PAYMENT without sender balance
    if row['type'] == 'PAYMENT' and row['oldbalanceOrg'] == 0 and row['amount'] > 0:
        return 0

    return 1


fraud['is_logical'] = fraud.apply(check_logical, axis=1)

fraud.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_error,is_full_drain,is_logical
0,182,CASH_IN,100328.70,C197678862,3389602.37,3489931.07,C386191921,219735.14,119406.44,0,0,-200657.40,0,0
1,210,PAYMENT,11660.99,C2009511954,94054.00,82393.01,M564284134,0.00,0.00,0,0,0.00,0,1
2,403,CASH_IN,230751.30,C1132312861,23117012.10,23347763.39,C1025694734,951294.81,720543.51,0,0,-461502.59,0,0
3,328,PAYMENT,17833.60,C1191709365,0.00,0.00,M293024801,0.00,0.00,0,0,-17833.60,1,0
4,563,CASH_OUT,246476.67,C342438889,0.00,0.00,C2087488957,4226850.16,4473326.82,0,0,-246476.67,1,0


## Encoding Categorical Variables

Machine learning algorithms require numerical inputs, therefore categorical variables must be converted into numerical representations.

The transaction type (`type`) contains categories such as CASH_OUT, TRANSFER, PAYMENT, and DEBIT. One-hot encoding was applied to convert these categories into binary numerical features while avoiding artificial ordering between categories.

In [ ]:
fraud = pd.get_dummies(
    fraud,
    columns=['type'],
    drop_first=True
)

fraud.head()

fraud = fraud.astype(int)

fraud.dtypes

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_error,is_full_drain,is_logical,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,182,100328.70,C197678862,3389602.37,3489931.07,C386191921,219735.14,119406.44,0,0,-200657.40,0,0,False,False,False,False
1,210,11660.99,C2009511954,94054.00,82393.01,M564284134,0.00,0.00,0,0,0.00,0,1,False,False,True,False
2,403,230751.30,C1132312861,23117012.10,23347763.39,C1025694734,951294.81,720543.51,0,0,-461502.59,0,0,False,False,False,False
3,328,17833.60,C1191709365,0.00,0.00,M293024801,0.00,0.00,0,0,-17833.60,1,0,False,False,True,False
4,563,246476.67,C342438889,0.00,0.00,C2087488957,4226850.16,4473326.82,0,0,-246476.67,1,0,True,False,False,False


## Removing Unnecessary Features

Transaction identifiers (`nameOrig` and `nameDest`) were removed because they represent unique IDs rather than meaningful transaction behaviour.

The `isFlaggedFraud` feature was removed because it is an existing detection flag that could introduce bias into the model.

In [5]:
fraud = fraud.drop(
    columns=[
        'nameOrig',
        'nameDest',
        'isFlaggedFraud'
    ]
)

fraud.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_error,is_full_drain,is_logical,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,182,100328.70,3389602.37,3489931.07,219735.14,119406.44,0,-200657.40,0,0,False,False,False,False
1,210,11660.99,94054.00,82393.01,0.00,0.00,0,0.00,0,1,False,False,True,False
2,403,230751.30,23117012.10,23347763.39,951294.81,720543.51,0,-461502.59,0,0,False,False,False,False
3,328,17833.60,0.00,0.00,0.00,0.00,0,-17833.60,1,0,False,False,True,False
4,563,246476.67,0.00,0.00,4226850.16,4473326.82,0,-246476.67,1,0,True,False,False,False


## Feature Importance Analysis

Random Forest was used to evaluate feature importance before model training.

This helps identify which variables contribute most to fraud detection and provides insight into the factors influencing fraudulent transactions.

In [6]:
from sklearn.ensemble import RandomForestClassifier

X = fraud.drop('isFraud', axis=1)
y = fraud['isFraud']


rf_feature = RandomForestClassifier(
    random_state=42
)

rf_feature.fit(X, y)


feature_importance = pd.Series(
    rf_feature.feature_importances_,
    index=X.columns
)

feature_importance.sort_values(
    ascending=False
)

is_full_drain     0.298199
newbalanceOrig    0.274427
balance_error     0.106796
is_logical        0.081295
amount            0.066979
newbalanceDest    0.057912
oldbalanceOrg     0.055356
step              0.026111
type_TRANSFER     0.012855
oldbalanceDest    0.010469
type_CASH_OUT     0.005680
type_PAYMENT      0.003917
type_DEBIT        0.000004
dtype: float64

In [7]:
fraud = fraud.drop(
    columns=[
        'type_DEBIT',
        'type_PAYMENT'
    ]
)

## Train-Test Split

The dataset was divided into training and testing sets to evaluate model performance on unseen data.

Stratified splitting was used because fraud detection datasets are highly imbalanced. This ensures that both training and testing sets maintain a similar proportion of fraudulent and non-fraudulent transactions.

In [8]:
from sklearn.model_selection import train_test_split

X = fraud.drop('isFraud', axis=1)
y = fraud['isFraud']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Random Forest Classification

Random Forest was used as a baseline model because it can handle complex relationships in structured transaction data. The model combines multiple decision trees to improve prediction performance and reduce overfitting.

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(X_test)

In [10]:
from sklearn.metrics import confusion_matrix, classification_report

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Confusion Matrix:
[[127083      0]
 [     1    169]]

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    127083
           1       1.00      0.99      1.00       170

    accuracy                           1.00    127253
   macro avg       1.00      1.00      1.00    127253
weighted avg       1.00      1.00      1.00    127253



## XGBoost Classification

XGBoost was selected as the final model because it is effective for structured datasets and can handle imbalanced classification problems by assigning more importance to the minority fraud class.

In [11]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(X_test)

In [12]:
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_pred))

XGBoost Confusion Matrix:
[[127083      0]
 [     0    170]]

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    127083
           1       1.00      1.00      1.00       170

    accuracy                           1.00    127253
   macro avg       1.00      1.00      1.00    127253
weighted avg       1.00      1.00      1.00    127253



## Model Performance Observation

Both Random Forest and XGBoost achieved near-perfect performance on the test data. This is mainly due to the strong fraud patterns present in the dataset, where features such as balance changes, full account drainage, and transaction behaviour provide clear separation between fraudulent and non-fraudulent transactions.

The similar performance between the two models suggests that the dataset itself is highly predictable rather than the result being caused by model complexity. Although the results are strong, the dataset may not fully represent real-world fraud scenarios where fraudulent behaviour is usually more complex and difficult to distinguish.